<a href="https://colab.research.google.com/github/ionikim/epilepsy_pediatrics_EEG/blob/main/src/01_loading/preprocessing_180326_filtering_add.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this file, we
1. automate the data preprocessing and
2. create multiplex horizontal visibility graphs for a certain time window in a file (edge list + adjacency matrix)

Comments:
- For the code to work, it needs to be in the same folder as the .edf and .seizure files.
- Window size can be adjusted; in my case the computer didn't have enough RAM for more than 5s.

What is missing:
- the code does not process files without a corresponding .seizure file
- actual graph creation for the correct windows (one wictal and one interictal)

In [8]:
!pip install ts2vg
!pip install mne
!git clone https://github.com/ionikim/epilepsy_pediatrics_EEG
from pathlib import Path
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
from ts2vg import HorizontalVG
import scipy.sparse as sp

Cloning into 'epilepsy_pediatrics_EEG'...
remote: Enumerating objects: 253, done.
remote: Counting objects: 100% (105/105), done.
remote: Compressing objects: 100% (99/99), done.
remote: Total 253 (delta 48), reused 8 (delta 6), pack-reused 148 (from 1)
Receiving objects: 100% (253/253), 27.57 MiB | 17.34 MiB/s, done.
Resolving deltas: 100% (103/103), done.


In [18]:
RAW_DIR = Path('epilepsy_pediatrics_EEG/data/raw')
print("RAW_DIR exists:", RAW_DIR.exists())

for f in RAW_DIR.iterdir():
   print("-", f.name)

RAW_DIR exists: True
- .gitkeep
- chb01_03.edf
- chb01_03.edf.seizures


In [20]:
# Extract seizure start and length from a .seizures annotation file.
def get_seizure_period(annotation_file):
    with open(annotation_file, "rb") as f:
        byte_array = f.read()
    start = int(bin(byte_array[38])[2:] + bin(byte_array[41])[2:], 2)
    length = byte_array[49]
    return start, length

# Load an EDF file, create a DataFrame, and label seizure periods
def process_edf_with_labels(edf_file, seizure_file):
    raw = mne.io.read_raw_edf(str(edf_file), preload=True, verbose=False)
    raw.filter(1., 45.) # band pass filtering
    raw.set_eeg_reference('average') # egg_re_reference

    electrode_names = raw.ch_names
    sfreq           = raw.info['sfreq']

    electrode_names = raw.ch_names
    sfreq           = raw.info['sfreq']
    n_samples       = len(raw.times)
    time_marks      = np.arange(n_samples) / sfreq
    electrode_data  = raw.get_data()

    df = pd.DataFrame(electrode_data.T, columns=electrode_names)
    df.index = time_marks
    df.index.name = 'Time (s)'

    # label
    if seizure_file:
        seizure_start, seizure_length = get_seizure_period(seizure_file)
        seizure_end = seizure_start + seizure_length
        df["label"] = df.index.map(
            lambda t: "ictal" if seizure_start <= t <= seizure_end else "interictal"
        )
        has_seizure = True
    else:
        df["label"] = "interictal"
        has_seizure = False

    return df, has_seizure


# extract a window of exactly `window_size_s` seconds from the DataFrame
# if df cointains a seizure: window contains only ictal timepoints, if no seizure: random position
def extract_window(df, window_size_s=10, mode="ictal"):
    sfreq = int(round(1 / (df.index[1] - df.index[0])))
    window_size_samples = window_size_s * sfreq

    if mode == "ictal":
        indices = np.where(df["label"] == "ictal")[0]
    elif mode == "interictal":
        indices = np.where(df["label"] == "interictal")[0]
    else:
        raise ValueError("mode must be 'ictal' or 'interictal'")

    valid_starts = [
        idx for idx in indices
        if idx + window_size_samples <= len(df)
        and (df["label"].iloc[idx : idx + window_size_samples] == mode).all()
    ]

    if not valid_starts:
        raise ValueError(f"No contiguous {mode} block of {window_size_s}s found.")

    start_idx = int(np.random.choice(valid_starts))
    window = df.iloc[start_idx : start_idx + window_size_samples]
    return window


# loop aover all .seizures files, process each edf and extract labelled windows
# returns: all_dataframes (dict of full labelled df's, windows (dict of extracted windows)
def process_all_files(pattern="*.seizures", window_size_s=10):
    seizure_files = sorted(glob.glob(pattern))

    if not seizure_files:
        print(f"No files found matching: {pattern}")
        return {}, {}

    all_dataframes = {}
    windows = {}

    for i, seizure_file in enumerate(seizure_files):
        edf_file = seizure_file.replace(".seizures", "")

        try:
            # 1. full dataframe
            df, has_seizure = process_edf_with_labels(edf_file, seizure_file)
            all_dataframes[edf_file] = df

            # 2. ictal / interictal window
            ictal_window = extract_window(df, window_size_s=window_size_s, mode="ictal")
            interictal_window = extract_window(df, window_size_s=window_size_s, mode="interictal")

            # 3. windows dict
            ictal_name = f"window_ictal_{i}"
            interictal_name = f"window_interictal_{i}"

            windows[ictal_name] = ictal_window
            windows[interictal_name] = interictal_window

            # 4. summary
            print(f"\n[{i}] {edf_file}")
            print(f"  Has seizure          : {has_seizure}")

            print(f"  Ictal window name    : {ictal_name}")
            print(f"  Ictal window size    : {len(ictal_window)} samples ({window_size_s}s)")
            print(f"  Ictal time range     : {ictal_window.index[0]:.1f}s → {ictal_window.index[-1]:.1f}s")
            print(f"  Ictal labels         : {ictal_window['label'].value_counts().to_dict()}")

            print(f"  Interictal name      : {interictal_name}")
            print(f"  Interictal size      : {len(interictal_window)} samples ({window_size_s}s)")
            print(f"  Interictal time range: {interictal_window.index[0]:.1f}s → {interictal_window.index[-1]:.1f}s")
            print(f"  Interictal labels    : {interictal_window['label'].value_counts().to_dict()}")

        except FileNotFoundError:
            print(f"ERROR: EDF file not found for {seizure_file} — expected {edf_file}")
        except ValueError as e:
            print(f"WARNING [{edf_file}]: {e}")
        except Exception as e:
            print(f"ERROR processing {seizure_file}: {e}")

    return all_dataframes, windows

In [21]:
# Run it
all_dataframes, windows = process_all_files(str(RAW_DIR / "*.seizures"), window_size_s=5)

# Access individual windows
# windows["window_ictal_0"]
# windows["window_interictal_2"]

# Check if all windows have the same number of samples
sizes = {name: len(df) for name, df in windows.items()}
print("\nWindow sizes:", sizes)

/tmp/ipykernel_168/3281475279.py:11: RuntimeWarning: Channel names are not unique, found duplicates for: {'T8-P8'}. Applying running numbers for duplicates.
  raw = mne.io.read_raw_edf(str(edf_file), preload=True, verbose=False)


Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filter length: 845 samples (3.301 s)

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.

[0] epilepsy_pediatrics_EEG/data/raw/chb01_03.edf
  Has seizure          : True
  Ictal window name    : window_ictal_0
  Ictal window size    : 1280 samples (5s)
  Ictal time range     : 3021.3s → 3026.3s
  Ictal labels         : {'ictal': 1280}
  Interictal name      : window_interictal_0
  Interictal size      : 1280 sampl

In [22]:
display(windows["window_ictal_0"])
display(windows["window_interictal_0"])

,FP1-F7,F7-T7,T7-P7,P7-O1,FP1-F3,F3-C3,C3-P3,P3-O1,FP2-F4,F4-C4,...,T8-P8-0,P8-O2,FZ-CZ,CZ-PZ,P7-T7,T7-FT9,FT9-FT10,FT10-T8,T8-P8-1,label
Time (s),,,,,,,,,,,,,,,,,,,,,
3021.339844,0.000080,0.000027,0.000088,-0.000090,-2.100539e-08,0.000069,-4.452070e-05,0.000079,-0.000082,0.000112,...,-0.000118,-0.000005,0.000047,-0.000015,-0.000159,-0.000022,-0.000109,0.000029,-0.000118,ictal
3021.343750,0.000092,0.000038,0.000088,-0.000101,2.633865e-05,0.000050,-3.497512e-05,0.000074,-0.000087,0.000131,...,-0.000107,0.000005,0.000048,-0.000017,-0.000160,-0.000034,-0.000121,0.000011,-0.000107,ictal
3021.347656,0.000095,0.000044,0.000086,-0.000104,2.192520e-05,0.000045,-2.191616e-05,0.000076,-0.000103,0.000134,...,-0.000024,0.000002,0.000052,-0.000016,-0.000155,-0.000042,-0.000150,-0.000041,-0.000024,ictal
3021.351562,0.000090,0.000040,0.000090,-0.000100,-1.292359e-05,0.000058,-7.593781e-06,0.000081,-0.000138,0.000139,...,0.000098,-0.000007,0.000062,-0.000012,-0.000148,-0.000044,-0.000197,-0.000092,0.000098,ictal
3021.355469,0.000083,0.000027,0.000101,-0.000095,-4.695206e-05,0.000079,6.169439e-07,0.000084,-0.000178,0.000152,...,0.000173,-0.000005,0.000073,-0.000007,-0.000146,-0.000038,-0.000246,-0.000099,0.000173,ictal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3026.320312,-0.000132,-0.000085,0.000053,0.000130,1.749529e-05,-0.000189,3.263073e-05,0.000104,0.000129,-0.000366,...,-0.000143,0.000315,-0.000275,0.000050,0.000284,0.000268,0.000182,0.000026,-0.000143,ictal
3026.324219,-0.000155,-0.000084,0.000057,0.000138,-2.145249e-05,-0.000183,4.247781e-05,0.000117,0.000112,-0.000353,...,-0.000117,0.000292,-0.000290,0.000063,0.000270,0.000256,0.000194,0.000029,-0.000117,ictal
3026.328125,-0.000177,-0.000087,0.000061,0.000147,-6.029228e-05,-0.000174,5.059908e-05,0.000126,0.000098,-0.000342,...,-0.000075,0.000267,-0.000299,0.000077,0.000254,0.000251,0.000203,0.000012,-0.000075,ictal


,FP1-F7,F7-T7,T7-P7,P7-O1,FP1-F3,F3-C3,C3-P3,P3-O1,FP2-F4,F4-C4,...,T8-P8-0,P8-O2,FZ-CZ,CZ-PZ,P7-T7,T7-FT9,FT9-FT10,FT10-T8,T8-P8-1,label
Time (s),,,,,,,,,,,,,,,,,,,,,
1868.164062,-0.000025,0.000013,0.000017,0.000018,-0.000004,-5.494549e-07,0.000002,0.000026,-1.969558e-06,-0.000017,...,-5.774653e-07,6.986619e-06,0.000009,-4.081439e-05,-0.000008,-2.044907e-05,0.000038,3.091802e-06,-5.774653e-07,interictal
1868.167969,-0.000026,0.000011,0.000010,0.000022,-0.000007,-5.810220e-06,0.000003,0.000028,8.513055e-07,-0.000018,...,-4.336241e-06,8.168050e-07,0.000009,-4.645026e-05,0.000003,-2.027838e-05,0.000052,5.784834e-06,-4.336241e-06,interictal
1868.171875,-0.000022,0.000011,0.000005,0.000022,-0.000004,-5.376945e-06,0.000003,0.000023,4.228540e-06,-0.000019,...,-7.739283e-06,3.782402e-08,0.000005,-4.729898e-05,0.000011,-1.300701e-05,0.000053,3.733957e-06,-7.739283e-06,interictal
1868.175781,-0.000017,0.000011,0.000003,0.000019,0.000003,-2.062120e-06,0.000001,0.000015,6.486336e-06,-0.000021,...,-1.148503e-05,5.299479e-06,-0.000002,-4.462683e-05,0.000015,-8.243080e-07,0.000045,-7.026918e-07,-1.148503e-05,interictal
1868.179688,-0.000011,0.000011,0.000003,0.000016,0.000012,-1.665832e-07,-0.000003,0.000011,6.449246e-06,-0.000023,...,-1.518232e-05,1.226411e-05,-0.000009,-4.101810e-05,0.000017,9.409208e-06,0.000037,-3.817514e-06,-1.518232e-05,interictal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1873.144531,0.000003,-0.000021,-0.000008,0.000030,-0.000038,-4.965723e-05,0.000018,0.000074,-1.286887e-05,-0.000056,...,-1.651296e-05,6.885516e-06,-0.000047,3.844024e-06,0.000052,6.856377e-05,0.000036,-2.182495e-05,-1.651296e-05,interictal
1873.148438,0.000007,-0.000020,-0.000006,0.000024,-0.000042,-4.390731e-05,0.000017,0.000072,-1.249733e-05,-0.000050,...,-2.120277e-05,2.547033e-06,-0.000042,3.866110e-06,0.000052,7.170113e-05,0.000036,-1.933682e-05,-2.120277e-05,interictal
1873.152344,0.000009,-0.000020,-0.000005,0.000020,-0.000045,-3.816678e-05,0.000016,0.000069,-1.054714e-05,-0.000044,...,-2.599651e-05,-1.284114e-06,-0.000035,2.378181e-06,0.000051,7.274854e-05,0.000036,-1.531350e-05,-2.599651e-05,interictal


In [23]:
# build a multiplex horizontal visibility graph from an EEG window df
# -> each electrode is on one layer, each time point connects all electrodes (inter-layer edges)
# returns: edge list, adjacency matrix, node_index (dict for global node index), layer_info (dict with per-electrode HVG info)

def build_multiplex_hvg(window_df):
    electrode_cols = [c for c in window_df.columns if c != "label"]
    n_electrodes   = len(electrode_cols)
    n_timepoints   = len(window_df)
    n_nodes_total  = n_electrodes * n_timepoints

    print(f"Electrodes   : {n_electrodes}")
    print(f"Time points  : {n_timepoints}")
    print(f"Total nodes  : {n_nodes_total}")

    # 1. NODE INDEX: map (electrode_idx, time_idx) -> global node index
    # node layout: electrode 0 gets nodes 0..n_timepoints-1,
    #              electrode 1 gets nodes n_timepoints..2*n_timepoints-1, etc.

    def node_id(layer, t):
        return layer * n_timepoints + t

    node_index = {
        (electrode_cols[l], t): node_id(l, t)
        for l in range(n_electrodes)
        for t in range(n_timepoints)
    }

    # 2. INTRA-LAYER EDGES: HVG edges within each electrode
    intra_edges = []
    layer_info  = {}

    for l, electrode in enumerate(electrode_cols):
        ts  = window_df[electrode].values
        hvg = HorizontalVG()
        hvg.build(ts)

        for (t_i, t_j) in hvg.edges:
            u = node_id(l, t_i)
            v = node_id(l, t_j)
            intra_edges.append({
                "source"       : u,
                "target"       : v,
                "source_label" : f"{electrode}_t{t_i}",
                "target_label" : f"{electrode}_t{t_j}",
                "edge_type"    : "intra",
                "layer"        : electrode,
            })

        layer_info[electrode] = {
            "n_intra_edges": len(hvg.edges),
            "layer_index"  : l,
        }

    print(f"Intra-layer edges : {len(intra_edges)}")

    # 3. INTER-LAYER EDGES: connect same time point across all electrodes
    # at each time point t, connect every electrode to every other (full coupling)

    inter_edges = []

    for t in range(n_timepoints):
        for l_i in range(n_electrodes):
            for l_j in range(l_i + 1, n_electrodes):   # upper triangle only (undirected)
                u = node_id(l_i, t)
                v = node_id(l_j, t)
                inter_edges.append({
                    "source"       : u,
                    "target"       : v,
                    "source_label" : f"{electrode_cols[l_i]}_t{t}",
                    "target_label" : f"{electrode_cols[l_j]}_t{t}",
                    "edge_type"    : "inter",
                    "layer"        : f"{electrode_cols[l_i]}<->{electrode_cols[l_j]}",
                })

    print(f"Inter-layer edges : {len(inter_edges)}")

    # 4. COMBINED EDGE LIST
    edge_list = pd.DataFrame(intra_edges + inter_edges)
    print(f"Total edges       : {len(edge_list)}")

    # 5. ADJACENCY MATRIX
    # use sparse matrix for efficiency (n_nodes can be large)
    adj_sparse = sp.lil_matrix((n_nodes_total, n_nodes_total), dtype=np.int8)

    for _, row in edge_list.iterrows():
        u, v = int(row["source"]), int(row["target"])
        adj_sparse[u, v] = 1
        adj_sparse[v, u] = 1  # undirected

    # convert to labeled DataFrame
    node_labels = [
        f"{electrode_cols[l]}_t{t}"
        for l in range(n_electrodes)
        for t in range(n_timepoints)
    ]
    adj_matrix = pd.DataFrame(
        adj_sparse.toarray(),
        index   = node_labels,
        columns = node_labels
    )

    return edge_list, adj_matrix, node_index, layer_info


# Run on first ictal window
ictal_key = next(k for k in windows if "ictal" in k)
window    = windows[ictal_key]

edge_list, adj_matrix, node_index, layer_info = build_multiplex_hvg(window)

# inspect results
print("\n--- Edge list (first 10) ---")
display(edge_list.head(10))

print("\n--- Adjacency matrix (first 5x5) ---")
display(adj_matrix.iloc[:5, :5])

print("\n--- Layer info ---")
display(pd.DataFrame(layer_info).T)

# split edge list back into intra / inter if needed
intra = edge_list[edge_list["edge_type"] == "intra"]
inter = edge_list[edge_list["edge_type"] == "inter"]

Electrodes   : 23
Time points  : 1280
Total nodes  : 29440
Intra-layer edges : 57868
Inter-layer edges : 323840
Total edges       : 381708

--- Edge list (first 10) ---


,source,target,source_label,target_label,edge_type,layer
0,284,285,FP1-F7_t284,FP1-F7_t285,intra,FP1-F7
1,285,286,FP1-F7_t285,FP1-F7_t286,intra,FP1-F7
2,283,284,FP1-F7_t283,FP1-F7_t284,intra,FP1-F7
3,280,284,FP1-F7_t280,FP1-F7_t284,intra,FP1-F7
4,279,284,FP1-F7_t279,FP1-F7_t284,intra,FP1-F7
5,286,287,FP1-F7_t286,FP1-F7_t287,intra,FP1-F7
6,286,289,FP1-F7_t286,FP1-F7_t289,intra,FP1-F7
7,286,290,FP1-F7_t286,FP1-F7_t290,intra,FP1-F7
8,278,279,FP1-F7_t278,FP1-F7_t279,intra,FP1-F7
9,279,280,FP1-F7_t279,FP1-F7_t280,intra,FP1-F7



--- Adjacency matrix (first 5x5) ---


,FP1-F7_t0,FP1-F7_t1,FP1-F7_t2,FP1-F7_t3,FP1-F7_t4
FP1-F7_t0,0,1,0,0,0
FP1-F7_t1,1,0,1,0,0
FP1-F7_t2,0,1,0,1,0
FP1-F7_t3,0,0,1,0,1
FP1-F7_t4,0,0,0,1,0



--- Layer info ---


,n_intra_edges,layer_index
FP1-F7,2499,0
F7-T7,2503,1
T7-P7,2514,2
P7-O1,2516,3
FP1-F3,2510,4
F3-C3,2512,5
C3-P3,2522,6
P3-O1,2535,7
FP2-F4,2518,8
F4-C4,2488,9
